# Lecture 4a — Serving a Model over a REST API (FastAPI)
**AI Operations · Module 4 · CPU only · core path ≈ 15 min in class · full notebook ≈ 45 min**

**You will:** (1) wrap a trained model in a validated HTTP API, (2) prove the API rejects bad input, (3) measure where latency comes from, (4) load-test it and find its saturation point, (5) scale it out with more worker processes.

Everything runs on a laptop CPU. Cells marked **YOUR TURN** ask you to write code; a **Solution** cell follows each — try first.

## 0 · Setup — write the project files and train two model versions

In [ ]:
import sys, subprocess
print(sys.version.split()[0])
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fastapi>=0.110", "uvicorn>=0.30", "scikit-learn", "pandas", "joblib", "requests", "matplotlib"], check=False)
import os; os.makedirs("data", exist_ok=True)

In [ ]:
%%writefile train_models.py
"""
train_models.py  --  AIOps Module 4 (Model Serving)
Trains TWO versions of a customer-churn classifier on a synthetic, deterministic dataset.
  v1 = LogisticRegression (fast, simple baseline)      -> data/model_v1.joblib
  v2 = RandomForest       (slower, usually more accurate) -> data/model_v2.joblib
Two versions let us practise canary releases, blue-green switches and rollback later.
CPU only; runs in a few seconds.
"""
import json, argparse, numpy as np, pandas as pd, joblib
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

FEATURES = ["tenure_months", "monthly_charge", "support_calls", "late_payments",
            "data_usage_gb", "contract_months", "num_products", "age"]

def make_data(n=6000, seed=7):
    r = np.random.default_rng(seed)
    df = pd.DataFrame({
        "tenure_months": r.integers(1, 72, n),
        "monthly_charge": r.normal(65, 20, n).clip(15, 150).round(2),
        "support_calls": r.poisson(2, n),
        "late_payments": r.poisson(0.7, n),
        "data_usage_gb": r.gamma(3, 8, n).round(2),
        "contract_months": r.choice([1, 12, 24], n, p=[.5, .3, .2]),
        "num_products": r.integers(1, 6, n),
        "age": r.integers(18, 80, n),
    })
    z = (-2.0 - 0.02*df.tenure_months + 0.01*df.monthly_charge + 0.15*df.support_calls
         + 0.2*df.late_payments - 0.01*df.contract_months - 0.1*df.num_products
         + 2.2*((df.support_calls >= 4) & (df.contract_months == 1))      # non-linear interactions:
         + 1.8*((df.tenure_months < 12) & (df.monthly_charge > 80))       # a linear model cannot
         + 1.5*((df.late_payments >= 2) & (df.num_products <= 2))         # capture these well
         - 1.6*((df.contract_months == 24) & (df.tenure_months > 36))
         + r.normal(0, .5, n))
    df["churn"] = (1/(1+np.exp(-z)) > r.uniform(size=n)).astype(int)
    return df

def main(out="data"):
    df = make_data()
    X, y = df[FEATURES], df["churn"]
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.25, random_state=0, stratify=y)
    models = {
        "v1": make_pipeline(StandardScaler(), LogisticRegression(max_iter=500)),
        "v2": RandomForestClassifier(n_estimators=150, max_depth=10, random_state=0, n_jobs=1),
    }
    report = {}
    for ver, m in models.items():
        m.fit(Xtr, ytr)
        acc = accuracy_score(yte, m.predict(Xte))
        joblib.dump({"model": m, "feature_columns": FEATURES, "version": ver}, f"{out}/model_{ver}.joblib")
        report[ver] = round(acc, 4)
        print(f"{ver}: test accuracy = {acc:.4f}")
    df.to_csv(f"{out}/churn.csv", index=False)
    Xte.head(200).to_csv(f"{out}/sample_requests.csv", index=False)
    json.dump({"rows": Xte.head(3).to_dict(orient="records")}, open(f"{out}/sample_payload.json", "w"), indent=2)
    json.dump(report, open(f"{out}/metrics.json", "w"))

if __name__ == "__main__":
    ap = argparse.ArgumentParser(); ap.add_argument("--out", default="data"); main(ap.parse_args().out)

In [ ]:
%%writefile rest_app.py
"""
rest_app.py  --  AIOps Module 4, Topic 1: model serving over a REST API (FastAPI)
Run:  MODEL_PATH=data/model_v1.joblib uvicorn rest_app:app --port 8000
Endpoints
  GET  /healthz                      liveness/readiness probe
  GET  /v1/models/churn              metadata (version, features)
  POST /v1/models/churn:predict      {"instances":[{...feature dict...}, ...]}
"""
import os, time, socket, joblib
import pandas as pd
from typing import List
from fastapi import FastAPI, HTTPException, Request
from pydantic import BaseModel, Field

MODEL_PATH = os.getenv("MODEL_PATH", "data/model_v1.joblib")
bundle = joblib.load(MODEL_PATH)                 # loaded ONCE at startup, never per request
model, FEATURES, VERSION = bundle["model"], bundle["feature_columns"], bundle["version"]
POD = os.getenv("POD_NAME", socket.gethostname())

class Customer(BaseModel):                       # schema = contract with the client
    tenure_months: int = Field(ge=0, le=120)
    monthly_charge: float = Field(ge=0, le=500)
    support_calls: int = Field(ge=0)
    late_payments: int = Field(ge=0)
    data_usage_gb: float = Field(ge=0)
    contract_months: int = Field(ge=1, le=60)
    num_products: int = Field(ge=1, le=20)
    age: int = Field(ge=18, le=110)

class PredictRequest(BaseModel):
    instances: List[Customer] = Field(min_length=1, max_length=1000)

app = FastAPI(title="Churn model service")
STATS = {"requests": 0, "errors": 0}

@app.middleware("http")
async def timing(request: Request, call_next):   # cheap observability: latency header + counters
    t0 = time.perf_counter()
    resp = await call_next(request)
    resp.headers["X-Process-Time-ms"] = f"{(time.perf_counter()-t0)*1000:.2f}"
    STATS["requests"] += 1
    STATS["errors"] += int(resp.status_code >= 400)
    return resp

@app.get("/healthz")
def healthz():
    return {"status": "ok", "model_version": VERSION}

@app.get("/v1/models/churn")
def metadata():
    return {"name": "churn", "version": VERSION, "features": FEATURES, "served_by": POD}

@app.get("/stats")
def stats():
    return {**STATS, "served_by": POD, "model_version": VERSION}

@app.post("/v1/models/churn:predict")
def predict(req: PredictRequest):
    # plain `def` => FastAPI runs it in a thread pool, so CPU-bound sklearn does not block the event loop
    try:
        X = pd.DataFrame([c.model_dump() for c in req.instances])[FEATURES]
        proba = model.predict_proba(X)[:, 1]
    except Exception as e:                       # never leak a stack trace to the client
        raise HTTPException(status_code=500, detail=f"inference failed: {type(e).__name__}")
    return {"predictions": [int(p >= 0.5) for p in proba],
            "probabilities": [round(float(p), 4) for p in proba],
            "model_version": VERSION, "served_by": POD}

In [ ]:
%%writefile load_test.py
"""
load_test.py -- generic closed-loop load generator for any model endpoint (REST / MLflow / Ray / K8s).
  python load_test.py --url http://localhost:8000/v1/models/churn:predict --format rest
  python load_test.py --url http://localhost:5001/invocations --format mlflow --concurrency 1 4 8
Reports p50/p95/p99 latency, throughput, errors and which replicas / model versions answered.
"""
import argparse, json, time, statistics, collections, concurrent.futures as cf
import pandas as pd, requests

def payload(rows, fmt, batch):
    recs = rows[:batch]
    if fmt == "rest":   return {"instances": recs}
    if fmt == "mlflow": return {"dataframe_records": recs}
    raise ValueError(fmt)

def one(sess, url, body):
    t0 = time.perf_counter()
    try:
        r = sess.post(url, json=body, timeout=30)
        ok = r.status_code == 200
        j = r.json() if ok else {}
        who = j.get("served_by", "n/a") if isinstance(j, dict) else "n/a"
        ver = j.get("model_version", "n/a") if isinstance(j, dict) else "n/a"
        if ver == "n/a" and isinstance(j, dict) and j.get("predictions") and isinstance(j["predictions"][0], dict):
            ver = j["predictions"][0].get("model_version", "n/a")      # MLflow pyfunc output
    except Exception:
        ok, who, ver = False, "n/a", "n/a"
    return (time.perf_counter() - t0) * 1000, ok, who, ver

def run(url, fmt, conc, duration, rows, batch):
    body = payload(rows, fmt, batch)
    lat, errs, who, ver = [], 0, collections.Counter(), collections.Counter()
    end = time.time() + duration
    def worker():
        s = requests.Session(); out = []
        while time.time() < end:
            out.append(one(s, url, body))
        return out
    with cf.ThreadPoolExecutor(conc) as ex:
        for res in [f.result() for f in [ex.submit(worker) for _ in range(conc)]]:
            for ms, ok, w, v in res:
                lat.append(ms); errs += (not ok); who[w] += 1; ver[v] += 1
    lat.sort(); n = len(lat); q = lambda p: lat[min(n-1, int(p*n))]
    return dict(concurrency=conc, requests=n, rps=round(n/duration, 1), p50_ms=round(q(.5), 1),
                p95_ms=round(q(.95), 1), p99_ms=round(q(.99), 1), errors=errs,
                replicas=len(who), versions=dict(ver))

if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--url", required=True); ap.add_argument("--format", default="rest")
    ap.add_argument("--concurrency", type=int, nargs="+", default=[1, 4, 8])
    ap.add_argument("--duration", type=float, default=5); ap.add_argument("--batch", type=int, default=1)
    ap.add_argument("--data", default="data/sample_requests.csv"); ap.add_argument("--out")
    a = ap.parse_args()
    rows = pd.read_csv(a.data).to_dict(orient="records")
    results = [run(a.url, a.format, c, a.duration, rows, a.batch) for c in a.concurrency]
    print(pd.DataFrame(results).to_string(index=False))
    if a.out: json.dump(results, open(a.out, "w"), indent=2)

In [ ]:
%%writefile m4util.py
"""m4util.py -- tiny helpers shared by the Module 4 notebooks (start / wait for / stop server processes)."""
import subprocess, time, os, signal, sys, requests

_procs = []

def start(cmd, log="server.log", env=None):
    """Start a server as a child process; stdout/stderr go to a log file (read it when things break)."""
    e = {**os.environ, **(env or {})}
    f = open(log, "w")
    p = subprocess.Popen(cmd, shell=isinstance(cmd, str), stdout=f, stderr=subprocess.STDOUT, env=e,
                         preexec_fn=os.setsid)
    _procs.append(p)
    return p

def wait_ready(url, timeout=90):
    """Poll a URL until it answers 200 (readiness probe, exactly what Kubernetes does)."""
    t0 = time.time()
    while time.time() - t0 < timeout:
        try:
            if requests.get(url, timeout=2).status_code == 200:
                return round(time.time() - t0, 1)
        except Exception:
            pass
        time.sleep(0.5)
    raise TimeoutError(f"{url} not ready after {timeout}s -- check the log file")

def stop(p, timeout=10):
    """Stop one server: SIGTERM the whole process group (so uvicorn/gunicorn workers die too),
    then WAIT until the group is really gone -- otherwise a restart can 'succeed' against the old process."""
    try:
        pgid = os.getpgid(p.pid)
    except Exception:
        return
    try:
        os.killpg(pgid, signal.SIGTERM)
        t0 = time.time()
        while time.time() - t0 < timeout:
            try: os.killpg(pgid, 0)            # signal 0 = "does the group still exist?"
            except ProcessLookupError: break
            time.sleep(0.2)
        else:
            os.killpg(pgid, signal.SIGKILL)
    except ProcessLookupError:
        pass
    try: p.wait(timeout=2)
    except Exception: pass

def stop_all():
    for p in _procs: stop(p)
    _procs.clear()

In [ ]:
!python train_models.py

Let's visit what's contained by the model file.

In [ ]:
import joblib
bundle = joblib.load("data/model_v1.joblib")
print("Feature columns:", bundle['feature_columns'])
print("Version:", bundle['version'])
print("Model:")
bundle['model']

Two versions were trained: **v1** (logistic regression) and **v2** (random forest). Every later notebook reuses them. Note that the model is saved *together with its `feature_columns`* — the serving code must never guess column order.

## 1 · Start the service and call it

In [ ]:
import requests, json, pandas as pd, m4util
# execute the following command to start the REST API server in the background, and wait until it is ready to accept requests
srv = m4util.start("MODEL_PATH=data/model_v1.joblib python -m uvicorn rest_app:app --port 8000", log="rest.log")
print("ready after", m4util.wait_ready("http://localhost:8000/healthz"), "s")
BASE = "http://localhost:8000"
print(requests.get(f"{BASE}/healthz").json())
print(json.dumps(requests.get(f"{BASE}/v1/models/churn").json(), indent=2))

In [ ]:
rows = json.load(open("data/sample_payload.json"))["rows"]
r = requests.post(f"{BASE}/v1/models/churn:predict", json={"instances": rows})
print(r.status_code, r.headers["X-Process-Time-ms"], "ms server-side")
print(json.dumps(r.json(), indent=2))

**Read the response:** `probabilities` are the model scores, `predictions` apply a 0.5 threshold, `model_version` and `served_by` make every answer *traceable* — essential once several versions and replicas exist.

## 2 · The contract: what happens with bad input?
A serving API is a contract. Invalid requests must fail **fast, clearly, and with a 4xx** (client's fault) — never with a 500 or, worse, a silent wrong answer.

In [ ]:
good = rows[0]
cases = {
  "valid":                 {"instances": [good]},
  "negative tenure":       {"instances": [{**good, "tenure_months": -5}]},
  "missing field":         {"instances": [{k: v for k, v in good.items() if k != "age"}]},
  "string instead of num": {"instances": [{**good, "monthly_charge": "lots"}]},
  "empty list":            {"instances": []},
  "wrong top-level key":   {"rows": [good]},
  "age 500":               {"instances": [{**good, "age": 500}]},
}
out = []
for name, body in cases.items():
    r = requests.post(f"{BASE}/v1/models/churn:predict", json=body)
    detail = "" if r.ok else r.json()["detail"][0]["msg"] if isinstance(r.json().get("detail"), list) else r.text[:60]
    out.append((name, r.status_code, detail))
pd.DataFrame(out, columns=["case", "HTTP status", "reason"])

Every invalid case returns **422** with a machine-readable reason, and the model was never invoked. That validation is the pydantic `Customer` class in `rest_app.py` — read it again and note each `Field(ge=…, le=…)` rule is a *business assumption* written as code.

## 3 · Where does latency come from?
Compare the model alone, then the full HTTP round trip.

In [ ]:
import time, joblib, numpy as np
b = joblib.load("data/model_v1.joblib"); model, cols = b["model"], b["feature_columns"]
data = pd.read_csv("data/sample_requests.csv")

# `timeit` runs a function `n` times and returns the median execution time in milliseconds.
def timeit(fn, n=30):
    ts = []
    for _ in range(n):
        t = time.perf_counter(); fn(); ts.append((time.perf_counter() - t) * 1000)
    return np.median(ts)

res = []
for bs in [1, 10, 100, 500, 1000]:
    X = data.sample(bs, replace=True, random_state=0); recs = X.to_dict(orient="records")
    local = timeit(lambda: model.predict_proba(X[cols]))
    sess = requests.Session()
    http = timeit(lambda: sess.post(f"{BASE}/v1/models/churn:predict", json={"instances": recs}))
    res.append(dict(batch=bs, model_only_ms=round(local, 2), http_total_ms=round(http, 2),
                    overhead_ms=round(http - local, 2), http_ms_per_row=round(http / bs, 3)))
pd.DataFrame(res)

**Observe:** a fixed per-request overhead (JSON parse, validation, network stack) dominates for single rows; batching amortizes it. The price of batching is that the *first* row waits for the *last* — a latency/throughput trade-off you will tune in the assignment.

## 4 · Load test — find the saturation point

In [ ]:
import load_test
rows_all = data.to_dict(orient="records")
URL = f"{BASE}/v1/models/churn:predict"
sweep = [load_test.run(URL, "rest", cc, 4, rows_all, 1) for cc in [1, 2, 4, 8, 16]]
df1 = pd.DataFrame(sweep)[["concurrency", "requests", "rps", "p50_ms", "p95_ms", "p99_ms", "errors"]]
df1

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].plot(df1.concurrency, df1.rps, "o-"); ax[0].set_title("Throughput (req/s)"); ax[0].set_xlabel("concurrent clients")
ax[1].plot(df1.concurrency, df1.p50_ms, "o-", label="p50"); ax[1].plot(df1.concurrency, df1.p95_ms, "s-", label="p95")
ax[1].set_title("Latency (ms)"); ax[1].set_xlabel("concurrent clients"); ax[1].legend()
plt.tight_layout(); plt.show()

**Read the curves.** Throughput rises, then flattens: once the single worker's CPU is saturated, extra clients only *queue*, so latency grows roughly linearly with concurrency (Little's Law: *concurrency ≈ throughput × latency*). The knee of the curve is your service's capacity — the number autoscalers and SLOs are built on.

## 5 · Scale out with more worker processes

In [ ]:
m4util.stop(srv)
srv2 = m4util.start("MODEL_PATH=data/model_v1.joblib python -m uvicorn rest_app:app --port 8001 --workers 2", log="rest2.log")
m4util.wait_ready("http://localhost:8001/healthz")
URL2 = "http://localhost:8001/v1/models/churn:predict"
sweep2 = [load_test.run(URL2, "rest", cc, 4, rows_all, 1) for cc in [1, 4, 8]]
cmp_ = pd.concat([df1[df1.concurrency.isin([1, 4, 8])].assign(workers=1),
                  pd.DataFrame(sweep2)[["concurrency", "requests", "rps", "p50_ms", "p95_ms", "p99_ms", "errors"]].assign(workers=2)])
cmp_.sort_values(["concurrency", "workers"])

**Honest reading:** on a 2-core laptop the *load generator itself* competes with the server for CPU, so you will not see a clean 2×. In production the client runs elsewhere. The lesson is the *mechanism*: workers (processes) add capacity because Python's GIL limits one process to one core of pure-Python work. The same idea scales further as **replicas** — Ray Serve replicas (Lecture 4c) and Kubernetes pods (Lecture 4d).

## 6 · YOUR TURN

**Task A — a resilient client.** Write `safe_predict(rows, retries=3)` that:
* retries on connection errors and **5xx** with exponential backoff (0.2 s, 0.4 s, …),
* does **not** retry on **4xx** (it will never succeed) and instead raises `ValueError` with the server's reason,
* uses a 5-second timeout.

Test it with (i) a good request, (ii) a bad request, and (iii) a dead port.

**Task B — pick a batch size.** Using your measurements from §3 (or new ones), find the largest batch size from `[1, 10, 100, 500, 1000]` whose HTTP round trip stays below **30 ms**, and report its rows/second.

In [ ]:
# Task A — write your code here
def safe_predict(rows, retries=3, base_url="http://localhost:8001"):   # 8001 = the 2-worker server still running
    raise NotImplementedError

**Solution A** (try first!)

In [ ]:
def safe_predict(rows, retries=3, base_url="http://localhost:8001"):   # 8001 = the 2-worker server still running
    delay = 0.2
    for attempt in range(retries + 1):
        try:
            r = requests.post(f"{base_url}/v1/models/churn:predict", json={"instances": rows}, timeout=5)
        except requests.RequestException as e:                   # network problem -> retry
            err = f"connection error: {type(e).__name__}"
        else:
            if r.status_code < 400:
                return r.json()
            if r.status_code < 500:                              # client error -> never retry
                raise ValueError(f"{r.status_code}: {r.json()['detail'][0]['msg']}")
            err = f"server error {r.status_code}"                # 5xx -> retry
        if attempt == retries:
            raise RuntimeError(f"gave up after {retries + 1} attempts ({err})")
        time.sleep(delay); delay *= 2

print(safe_predict(rows[:1])["predictions"])
try: safe_predict([{**rows[0], "age": 5}])
except ValueError as e: print("bad request ->", e)
t = time.time()
try: safe_predict(rows[:1], base_url="http://localhost:9")
except RuntimeError as e: print("dead port ->", e, f"(after {time.time()-t:.1f}s of backoff)")

In [ ]:
# Task B — write your code here

**Solution B**

In [ ]:
tbl = pd.DataFrame(res)
ok = tbl[tbl.http_total_ms < 15]
best = ok.loc[ok.batch.idxmax()]
print(f"largest batch under 15 ms: {int(best.batch)} rows -> {best.http_total_ms} ms -> {1000*best.batch/best.http_total_ms:.0f} rows/s")

## 7 · Clean up

In [ ]:
m4util.stop_all(); print("servers stopped")

### Recap
* A serving API = **model + schema + health endpoint + traceability fields**.
* Validate at the edge; return 422 not 500.
* Latency = fixed overhead + compute; batching trades latency for throughput.
* Load tests find the saturation knee; more workers/replicas move it right.

**Next:** Lecture 4b — let MLflow generate the API for you, and use its Registry to manage versions.